# Searching for absorbers and measuring outflow velocity

This notebook looks for stacked absorption lines and uses them to characterise outflow velocities

In [ ]:
import numpy as np
from astropy.io import fits
import astropy.table as aptb
import astropy.units as u
import matplotlib.pyplot as plt
import copy
from scipy.optimize import curve_fit

from tangelo import spectroscopy as tgspec
from tangelo import models as tgmod
from tangelo import fitting as tgfit
from tangelo import catalogue_operations as tgcat
from tangelo import constants as tgconst
from tangelo import plotting as tgplot
from tangelo import image_processing as tgimg
from tangelo import io as tgio
from tangelo import quality_control as tgqc

In [ ]:
# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub"

tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

In [ ]:
# List of emission and absorption lines to average together in each spectrum
emission_lines = tgconst.emlines
absorption_lines = tgconst.abslines
hi_abslines = ['SiIV1394', 'SiIV1403']
li_abslines = ['SiII1260', 'CII1334', 'SiII1527']
optthin_lines = ['HeII1640','OIII1666','CIII1907','CIII1909']

In [ ]:
# Some helper functions to be used later
def linfunc(x, a, b):
    return a * x + b
def flatten(spec):
    wave = np.arange(0, np.size(spec))
    popt, pcov = curve_fit(linfunc, wave, spec, p0=[0,0])
    return spec - linfunc(wave, popt[0], popt[1]) + popt[1]

def is_neggauss(popt, pcov, sigma = 1.0):
    lmin = tgmod.gaussian(popt[1], *popt)
    fluxmax = popt[0] + sigma * np.sqrt(pcov[0,0])
    lmin_max = tgmod.gaussian(popt[1], fluxmax, *popt[1:])
    return lmin < 0 and lmin_max < 0

def has_abs(line, row, sigma=3):
    """Check for a detected, un-flagged absorption line"""
    return (row[f"SNR_{line}"] < -sigma) and (not row[f"FLAG_{line}"])

In [ ]:
# Now average absorption lines for all sources, looking for statistically significant absorption (just like emission)
# here, we stack the low-ionization abs lines and high separately, then fit gaussians to each

if not 'DV_LI_ABS' in megatable.colnames: # See if columns already exist, if not, create them
    megatable.add_columns([aptb.Column(np.nan*np.zeros(len(megatable)), "DV_LI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "DV_LI_ABS_ERR"),
                                aptb.Column(np.array(['' for x in megatable], dtype='U255'), "DV_LI_ABS_LINES"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_LI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_LI_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_LI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_LI_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "DV_HI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "DV_HI_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_HI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_HI_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_HI_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_HI_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "DV_TOT_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "DV_TOT_ABS_ERR"),
                                aptb.Column(np.array(['' for x in megatable], dtype='U255'), "DV_TOT_ABS_LINES"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_TOT_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "W_TOT_ABS_ERR"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_TOT_ABS"),
                                aptb.Column(np.nan*np.zeros(len(megatable)), "EW_TOT_ABS_ERR"),])
else: # If they do exist, reset them
    megatable["DV_LI_ABS"] *= np.nan
    megatable["DV_LI_ABS_ERR"] *= np.nan
    megatable["DV_LI_ABS_LINES"] = np.array(['' for x in megatable], dtype='U255')
    megatable["W_LI_ABS"] *= np.nan
    megatable["W_LI_ABS_ERR"] *= np.nan
    megatable["EW_LI_ABS"] *= np.nan
    megatable["EW_LI_ABS_ERR"] *= np.nan
    megatable["DV_HI_ABS"] *= np.nan
    megatable["DV_HI_ABS_ERR"] *= np.nan
    megatable["W_HI_ABS"] *= np.nan
    megatable["W_HI_ABS_ERR"] *= np.nan
    megatable["EW_HI_ABS"] *= np.nan
    megatable["EW_HI_ABS_ERR"] *= np.nan
    megatable["DV_TOT_ABS"] *= np.nan
    megatable["DV_TOT_ABS_ERR"] *= np.nan
    megatable["DV_TOT_ABS_LINES"] = np.array(['' for x in megatable], dtype='U255')
    megatable["W_TOT_ABS"] *= np.nan
    megatable["W_TOT_ABS_ERR"] *= np.nan
    megatable["EW_TOT_ABS"] *= np.nan
    megatable["EW_TOT_ABS_ERR"] *= np.nan

absnames = ['LI', 'HI', 'TOT'] # Names of the absorption types
count = 0 # Initialize counter of successful absorbers found
sig_level = 3.0 # Significance level for detection
spec_half_width = 3000
search_reg_hw = 1000
vel_bin = 60
USE_MEASURED_VAR = False
test = [
#     'MACS0416NE P1154',
#     'MACS1206 P2893',
#     'BULLET P869',
#     'BULLET P873',
#     "MACS1206 P2861",
#     "SMACS2131 SM122",
#     "A2744 P3423",
#     "RXJ1347 P4556",
#     "A370 SP5523",
#     "A2744 SP5408",
#     "MACS0416NE P1172",
#     "MACS0940 M15",
#     "MACS0416S P8961",
#     "RXJ1347 SM1089"
#     "MACS0940 M56"
] # Test sources

peak_check_whitelist = [
    
]

N_MC = 1000 # Number of Monte Carlo iterations for error estimation

for i, row in enumerate(megatable):
    iden = row['iden']
    cluster = row['CLUSTER']
    idfrom = row['idfrom']
    fullname = f"{cluster} {iden}"
    
    special = f"{cluster} {iden}" == "RXJ1347 SP1208"

    if test: # Testing the code
        if fullname not in test:
            continue
    
    print(f"==========")
    print(f"\nProcessing {iden} from {cluster}...")

    # Skip source with such high redshift that not even the SiII1260 line is within the MUSE range
    if (row['LPEAKR'] / 1215.67) * tgconst.wavedict['SiII1260'] > 9350:
        print("Source redshift is too high for any lines to be in MUSE range. Skipping.")
        continue
    
    # If the source has no HST continuum detection and is right next to a low-z object, exclude it automatically
    if idfrom == 'MUSELET':
        clustermems = tgcat.find_closest_cluster_member(row, maxdist = 2.0 * u.arcsec, return_all=True)
        if len(clustermems) > 0:
            print("Cluster member(s) close to target with no HST detection. Excluding.")
            continue
    
    # Skip the source entirely when there's no detectable continuum -- these sources frequently give false positives
    # and shouldn't have measurable absorption anyway if there's no continuum
    contcenter = 1300 * (1 + row['z']) # Center of the continuum measurement
    specrng = 80 # Range around continuum center to measure continuum
    nbimg = tgimg.make_muse_img(row, 10., lcenter = contcenter, width = specrng) #make a cutout muse image
    imgcont, imgcont_err = nbimg.background(niter=1, sigma=3.0) #find "background" value (gently sigma clipping)
    if SPEC_TYPE == 'weight_skysub': # If the spectrum has already been sky-subtracted, set imgcont to zero
        imgcont = 0.0

    spectab = tgio.load_spec(cluster, iden, idfrom, spec_type=SPEC_TYPE, spec_source=SPEC_SOURCE) # Load the spectrum
    speccont = spectab['spec'][np.logical_and(contcenter - specrng < spectab['wave'],
                                              spectab['wave'] < contcenter + specrng)] #measure the "continuum" in the spectrum over the same range
    if np.nanmedian(speccont) - imgcont < sig_level * imgcont_err:
        # If the continuum is less than 3 sigma above background, exclude the source
        print('Source with no continuum found. Excluding.')
        # Plot the continuum along with lyman alpha narrowband contours for visual confirmation
        lfig, lax = plt.subplots(1,1,figsize=(5,5), facecolor='w')
        nbimg.plot(ax=lax, vmin=0, scale='sqrt')
        lya_nbimg = tgimg.make_muse_img(row, 10, lcenter=row['LPEAKR'], width=1.5, cont = (20, 5))
        lya_nbbkg = lya_nbimg.background(niter=5, sigma=2.5)
        lax.contour(lya_nbimg.data, levels=np.arange(2,11,1)*lya_nbbkg[1], colors='w')
        plt.show()
        plt.close()
        continue
    
    # Generate Lyman-alpha spectrum for comparison against absorption features
    lyvel, lyspec, lyspecerr = tgspec.avg_lines(row, ['LYALPHA'], absorption=False, velbounds=[-spec_half_width, spec_half_width],
                                                z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                               velstep=vel_bin)
    
    # Generate low-ionization spectra
    print(f"Generating low-ionization spectra...")
    lispecs = {} # a dictionary where we will keep all these spectra
    lilines = copy.deepcopy(li_abslines)
    for line in list(lilines): # FIX 1: iterate over a copy to avoid skipping elements when removing
        if (row["LPEAKR"] / 1215.67) * tgconst.wavedict[line] > 9350: # Exclude lines outside MUSE wavelength range
            print(f"{line} lies beyond MUSE wavelength range.")
            lilines.remove(line)
        elif line == 'SiII1527':
            continue # Never exclude SiII1527 as long as it's within range
        elif row[f"FLAG_{line}"] and row[f'FLAG_{line}'] != 'm': # Exclude flagged lines, except those that only failed matched filtering.
            print(f"{line} was previously flagged, excluding it.")
            lilines.remove(line)
    
    # Now go through lines in lilines one by one and make spectra
    # Starting with CII1334
    ciiabs = has_abs("CII1334", row) & ('CII1334' in lilines) # Check if CII1334 is detected and un-flagged 
    if ciiabs: # If so, extract a spectrum for it
        ciivel, ciispec, ciispecerr = tgspec.avg_lines(row, ['CII1334'], absorption=False, velbounds=[-spec_half_width, spec_half_width],
                                                       z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                                      velstep=vel_bin, use_measured_var=USE_MEASURED_VAR, flags=['c','s','d'])
        lispecs['CII'] = [ciivel, ciispec, ciispecerr, ['CII1334']] # Include information about what lines have been used
    remlines = [l for l in lilines if l != 'CII1334'] # What un-flagged lines are now left?

    # Next, SiII1260, which will be combined with SiII1527 if both are present
    if 'SiII1260' in remlines:
        sivel, sispec, sispecerr = tgspec.avg_lines(row, remlines, absorption=False, velbounds=[-spec_half_width, spec_half_width],
                                                    z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                                   velstep=vel_bin, use_measured_var=USE_MEASURED_VAR, flags=['c','s','d']) #stack them
        lispecs['SiII'] = [sivel, sispec, sispecerr, remlines]
    
    # Finally, make an overall low-ionization stack
    abvel, abspec, abspecerr = tgspec.avg_lines(row, lilines, absorption=False, velbounds=[-spec_half_width, spec_half_width],
                                            z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                               velstep=vel_bin, use_measured_var=USE_MEASURED_VAR, flags=['c','s','d']) #finally make an overall stack
    lispecs['LIALL'] = [abvel, abspec, abspecerr, lilines]
    
    # Generate high-ionization spectrum
    print(f"Generating HI spectra...")
    hilines = copy.deepcopy(hi_abslines)
    for line in list(hilines): # FIX 1: iterate over a copy to avoid skipping elements when removing
        if (row["LPEAKR"] / 1215.67) * tgconst.wavedict[line] > 9350: # FIX 3: exclude lines outside MUSE wavelength range
            print(f"{line} lies beyond MUSE wavelength range.")
            hilines.remove(line)
        elif row[f"FLAG_{line}"]:
            print(f"{line} was previously flagged, excluding it")
            hilines.remove(line)
    habvel, habspec, habspecerr = tgspec.avg_lines(row, hilines, absorption=False, velbounds=[-spec_half_width, spec_half_width],
                                               z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                                  velstep=vel_bin, flags=['c','s','d'])
    
    
    # Generate total absorption spectrum
    print(f"Generating TOTAL spectra...")
    vel, spec, specerr = tgspec.avg_lines(row, lilines+hilines, absorption=False, velbounds=[-spec_half_width,spec_half_width],
                                      z = row['LPEAKR'] / 1215.67 - 1, spec_source=SPEC_SOURCE, spec_type=SPEC_TYPE,
                                         velstep=vel_bin, use_measured_var=USE_MEASURED_VAR, flags=['c','s','d'])
    
    # Plot these out so we can see how they look
    # Spectrum plotting (Note, LI spectrum is plotted at the end because we don't know in advance which we will use)
    fig, ax = plt.subplots(4, 1, figsize=(5,5), facecolor='white', sharex=True)
    fig.subplots_adjust(hspace=0)
    ax[-1].set_xlabel(r'$v$ [km\,s$^{-1}$]')
    ax[-2].set_ylabel(r'$f_v$ [erg\,s$^{-1}$\,cm$^{-2}$\,(km\,s$^{-1}$)$^{-1}$]')
    ax[0].set_title(f"{cluster} ID {iden}: $ z = {row['z']:.4f} $")
    ax[0].plot(lyvel, lyspec, drawstyle='steps-mid', color='slateblue')
    ax[0].fill_between(lyvel, lyspec-lyspecerr, lyspec+lyspecerr, step='mid', facecolor='slateblue', alpha=0.3, linestyle='')
    tgplot.lya_mod_plot(row, ax[0])
    # If habspec is all-NaN, fill the plot with a gray hatched box
    if np.all(np.isnan(habspec)):
        ax[2].axhspan(0, 1, color='gray', alpha=0.5, hatch='//', label='No valid HI lines')
    else:
        ax[2].plot(habvel, habspec, drawstyle='steps-mid', color='darkorange', label='SiIV1394+1403')
        ax[2].fill_between(habvel, habspec-habspecerr, habspec+habspecerr, step='mid', facecolor='darkorange', alpha=0.3, linestyle='')
    ax[3].plot(vel, spec, drawstyle='steps-mid', color='black', label='All lines')
    ax[3].fill_between(vel, spec-specerr, spec+specerr, step='mid', facecolor='black', alpha=0.3, linestyle='')
    ax[2].legend()
    ax[3].legend()

    # Fitting
    print(f"Starting fitting routine...\n")
    fitreg = (-2000. < vel) * (vel < 2000.)
    fitreg &= ~np.isnan(vel)
    fitreg &= ~np.isnan(spec)
    fitreg &= ~np.isnan(specerr)
    fitreg &= spec != 0
    fitreg &= (specerr > 0)
    fitreg &= ~np.isinf(spec)
    fitreg &= ~np.isinf(specerr)
    if special:
        fitreg &= vel < 2200 # This ONE source needs special treatment because the spectrum contains a weird break in it.
    if np.sum(np.isnan(vel * spec * specerr)) > 0: # Stop if no fitting will happen anyway
        print("Source redshift is too high, no lines to fit.")
        continue
    bounds = [[-1e5, -search_reg_hw, 60, -1000, -np.inf],
              [1e5, search_reg_hw, 500, 1000, np.inf]]
    popts = {}
    perrs = {}
    psamps = {}
    
    # Generate a dictionary of all the spectra we want to fit, so we can loop over them more easily. 
    # Each entry is a list of [vel, spec, specerr]
    allspecs = lispecs | {'HI': [habvel, habspec, habspecerr]} | {'TOT': [vel, spec, specerr]}

    # Remove any spectra that are entirely NaN or zero in the fitting region, and skip fitting for those
    for spec_type, sp in list(allspecs.items()):
        if np.sum(np.isnan(sp[0][fitreg] * sp[1][fitreg] * sp[2][fitreg])) > 0:
            print(f"Source redshift is too high, no {spec_type} lines to fit.")
            popts[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            perrs[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            psamps[spec_type] = [[np.nan, np.nan, np.nan, np.nan, np.nan]]
            del allspecs[spec_type] # Remove this spectrum from the dictionary so it won't be plotted later

    # Loop over all spectral types, look for peaks, fit gaussians, and perform Monte Carlo error estimation
    for l, (spec_type, sp) in enumerate(allspecs.items()):
        # Generate arrays to be fitted
        fit_vel = sp[0][fitreg]
        fit_spec = sp[1][fitreg]
        fit_errs = np.abs(sp[2][fitreg])
        # Perform a preliminary fit to get starting parameters, trying several initial guesses for VPEAK
        fit_found = False
        vpeak_guesses = [0, -300, 300]
        # Update FLUX and CONST guesses based on this spectrum
        guesses = {
            'FLUX': -np.abs(np.min(sp[1]) - np.median(sp[1])) * np.max(np.ediff1d(sp[0])) * 15,
            'VPEAK': 0,
            'FWHM': 300,
            'CONST': np.median(sp[1]),
            'SLOPE': 0,
        }
        print(f"Performing initial fitting of {spec_type} spectrum...")
        for idx, vpeak_guess in enumerate(vpeak_guesses):
            guesses['VPEAK'] = vpeak_guess
            is_last_guess = (idx == len(vpeak_guesses) - 1)
            try:
                # Initial fitting to get starting parameters
                popt, pcov = curve_fit(tgmod.gaussian, fit_vel, fit_spec, sigma=fit_errs,
                                        p0 = list(guesses.values()), absolute_sigma=True, max_nfev=100000,
                                        bounds = bounds)
                if (idfrom == 'MUSELET' and popt[-2] > 0.5): # Check for possible contamination in MUSELET sources
                    print(f"Continuum level is implausibly high for a MUSELET source ({popt[-2]:.2f})." + 
                          ("" if is_last_guess else " Trying next VPEAK guess."))
                    continue
            except (RuntimeError, ValueError):
                print(f"Initial fit failed for {spec_type} spectrum with VPEAK guess {vpeak_guess}." +
                      ("" if is_last_guess else " Trying next VPEAK guess."))
                continue
            if popt[0] / np.sqrt(pcov[0,0]) > -3.0: # If no tentative absorption found, skip
                print(f"No tentative {spec_type} absorption found with VPEAK guess {vpeak_guess}." +
                      ("" if is_last_guess else " Trying next VPEAK guess."))
                continue
            # If tentative absorption is found, check if it's physically plausible (i.e. not a negative flux Gaussian)
            if is_neggauss(popt, pcov, sigma=1.0):
                # if SPEC_SOURCE == 'R21':
                #     print(f"Negative flux detected in R21 background-subtracted spectrum. Discarding fit.")
                #     continue
                print(f"Negative flux detected...")
                popt_p, pcov_p = curve_fit(tgmod.gaussian, fit_vel, np.maximum(fit_spec, 0), sigma=fit_errs,
                                        p0 = popt, absolute_sigma=True, max_nfev=100000,
                                        bounds = bounds)
                if np.abs(popt_p[0] / np.sqrt(pcov_p[0,0])) > 5:
                    # Good significant fit, proceed to MC
                    print(f"Still significant even with negative values disallowed.")
                    fit_found = True
                    break
                else:
                    print("Fit requires negative flux, checking corresponding image...")
                    if nbimg.background(niter=5, sigma=2.5)[0] < 0. and SPEC_TYPE != 'weight_skysub':
                        print("Negative background detected. Keeping fit.")
                        fit_found = True
                        break
                    else:
                        print("Background is positive or sky-subtracted spectra are being used." +
                              ("" if is_last_guess else " Trying next VPEAK guess."))
                        continue
            else:
                # Fit is good, significant, and doesn't involve negative flux - proceed to MC
                fit_found = True
                break
        
        # Check if a valid fit was found before proceeding to Monte Carlo
        if not fit_found:
            print(f"No valid preliminary fit found for {spec_type} spectrum after trying all VPEAK guesses.\n")
            popts[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            perrs[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            psamps[spec_type] = [[np.nan, np.nan, np.nan, np.nan, np.nan]]
            continue
        else:
            print(f"Preliminary fit found for {spec_type} spectrum with VPEAK guess {vpeak_guess}.\n")

        # Proceed to full Monte-Carlo fitting with error estimation and autocorrelation analysis
        pfit, psamp = tgfit.fit_mc(tgmod.gaussian, fit_vel, fit_spec, fit_errs, popt,
                                   niter=N_MC, chisq_thresh=500, sig_clip=np.inf, bounds=bounds, return_sample=True,
                                   autocorrelation=True, max_lag=10, baseline_order=1, errfunc='stddev_7')
        
        # Check for a peak-dominated fit (i.e. a fit where a single channel dominates, which are prone to systematics)
        if tgqc.check_peak_dominant(fit_vel, fit_spec, fit_errs, pfit[0][1], pfit[0][2], pfit[0][0], pfit[0][3]):
            print("Fitted line is dominated by a single channel. Skipping.\n")
            popts[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            perrs[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            psamps[spec_type] = [[np.nan, np.nan, np.nan, np.nan, np.nan]]
            continue
        
        popt = pfit[0]
        perr = pfit[1]
        # Check for multiple comparable peaks (systematics test)
        model_prelim = tgmod.gaussian(fit_vel, *popt)
        residuals_prelim = fit_spec - model_prelim
        cont_model = popt[-1] * (fit_vel - popt[1]) + popt[-2]
        cont_sub_spec = fit_spec - cont_model
        
        # Convert flux to peak amplitude for proper comparison
        # tgmod.gaussian takes FWHM directly, converts internally to sigma
        # sigma = fwhm / (2 * sqrt(2 * ln(2)))
        # amplitude = flux / (sigma * sqrt(2*pi))
        # Therefore: amplitude = flux * 2 * sqrt(2*ln(2)) / (fwhm * sqrt(2*pi))
        # Which simplifies to: amplitude = flux * 0.9394 / fwhm
        fitted_amplitude = popt[0] * 0.9394 / popt[2]
        fitted_fwhm = popt[2]  # Already FWHM
        fitted_fwhm_err = perr[2]

        mf_exclusion_regions = None
        if spec_type == 'HI': # Need to mask doublet partners (though not the central line) otherwise they will show up in matched filtering
            central_vel = popt[1]
            vel_offset = tgspec.wave2vel(tgconst.wavedict['SiIV1403'], tgconst.wavedict['SiIV1394'])
            mask_width = popt[2] / 2 # Mask width equal to the fitted FWHM
            exclusion_lower = (central_vel - vel_offset - mask_width, central_vel - vel_offset + mask_width)
            exclusion_upper = (central_vel + vel_offset - mask_width, central_vel + vel_offset + mask_width)
            mf_exclusion_regions = [exclusion_lower, exclusion_upper]
            ax[2].axvspan(exclusion_lower[0], exclusion_lower[1], color='red', alpha=0.3, label='Masked region')
            ax[2].axvspan(exclusion_upper[0], exclusion_upper[1], color='red', alpha=0.3)
            ax[2].legend()

        # Check matched filter significance
        mf_reports = {}
        mf_widths = [max(0.1 * popt[2], popt[2] - perr[2]), popt[2], popt[2] + perr[2]]

        for width in mf_widths:
            report = tgfit.matched_filter_significance(
                        fit_vel,  # FIX 4: use this spectrum's own velocity axis
                        cont_sub_spec,
                        fit_errs,
                        popt[1],    # centre
                        width,    # fwhm
                        popt[0],    # flux -- negative for absorption, positive for emission; sign is all that matters
                        exclude_window_fwhm=1.0,
                        exclude_regions=mf_exclusion_regions
            )
            mf_reports[width] = report

        # Choose the worst report
        mf_report = mf_reports[min(mf_reports, key=lambda k: mf_reports[k]['parametric_sigma'])]
        
        if mf_report['parametric_sigma'] < 3: # Less than 3sigma significance in matched filter test
            print(f"Matched filter test failed with percentile {mf_report['percentile']:.3f}.")
            print(f"Skipping.\n")
            popts[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            perrs[spec_type] = [np.nan, np.nan, np.nan, np.nan, np.nan]
            psamps[spec_type] = [[np.nan, np.nan, np.nan, np.nan, np.nan]]
            continue
        else:
            print(f"Matched filter test passed with percentile {mf_report['percentile']:.3f}.")
        
        popts[spec_type] = pfit[0]
        perrs[spec_type] = pfit[1]
        psamps[spec_type] = psamp
        
        print(f"found {popts[spec_type][0]/perrs[spec_type][0]:.1f}sigma {spec_type} abs at {popt[1]:.2f}km/s.\n" if popts[spec_type][0]/perrs[spec_type][0] < -sig_level \
                  else f"{spec_type} absorption not significant.\n")
    
    # Now find which of the LI specs gives the best result
    likeys = [k for k in allspecs.keys() if k not in ('HI', 'TOT')] # FIX 7: explicit LI key selection
    best_lispec = None
    if len(likeys) > 0:
        best_lispec = min(likeys, key=lambda k: popts[k][0] / perrs[k][0] if not np.isnan(popts[k][0]) else float('inf')) # Finds the maximum significance
    
    # Plot the best LI spectrum in ax[1]
    if best_lispec is not None:
        ax[1].plot(allspecs[best_lispec][0], allspecs[best_lispec][1], drawstyle='steps-mid', 
                   color='teal', label = '+'.join(lispecs[best_lispec][-1]))
        ax[1].fill_between(allspecs[best_lispec][0], allspecs[best_lispec][1] - allspecs[best_lispec][2], 
                           allspecs[best_lispec][1] + allspecs[best_lispec][2], step='mid', facecolor='teal', 
                           alpha=0.3, linestyle='')
        ax[1].legend()
    else:
        ax[1].axhspan(0, 1, color='gray', alpha=0.5, hatch='//', label='No valid LI lines')
        ax[1].legend()
    
    # Plot all the models (axes fixed: LI=ax[1], HI=ax[2], TOT=ax[3])
    velhires = np.linspace(vel[0], vel[-1], 1000)
    for _ax, _fkey in [(ax[1], best_lispec), (ax[2], 'HI'), (ax[3], 'TOT')]:
        if _fkey is None:
            continue
        allfits = np.array([tgmod.gaussian(velhires, *p) for p in psamps[_fkey]])
        downpopt = np.percentile(allfits, 16.0, axis=0)
        uppopt = np.percentile(allfits, 84.0, axis=0)
        _ax.fill_between(velhires, downpopt, uppopt,
                         linestyle='', color='fuchsia' if popts[_fkey][0] / perrs[_fkey][0] < -3.0 else 'coral', alpha=0.6,
                         label='model')
        
    plot_dir = tgio.get_plot_dir(cluster, iden)
    plt.savefig(plot_dir / f"{cluster}_{iden}_stacked_absorption.png", dpi=300)
    
    plt.show()
    plt.close()
    
    # Enter results into the megatab
    if best_lispec is not None:
        popts['LI'] = popts[best_lispec]
        perrs['LI'] = perrs[best_lispec]
        psamps['LI'] = psamps[best_lispec]
    else:
        popts['LI'] = [np.nan, np.nan, np.nan, np.nan, np.nan]
        perrs['LI'] = [np.nan, np.nan, np.nan, np.nan, np.nan]
        psamps['LI'] = [[np.nan, np.nan, np.nan, np.nan, np.nan]]
    goods = np.array([False, False, False])
    percentile = 100 - 0.5 * (100 - tgfit.sigma_to_percentile(sig_level))
    for l, abstype in enumerate(['LI', 'HI', 'TOT']):
        significant = popts[abstype][0] / perrs[abstype][0] < -sig_level if N_MC < 5000 else np.percentile(np.array(psamps[abstype])[:,0], percentile) < 0
         # For smaller MC runs, use the standard error estimation; for larger runs, use the distribution of samples to robustly determine significance, 
         # which is more reliable for non-Gaussian distributions
        if significant: # robustly ensures significant fit at 3 sigma equivalent level
            print(f"Entering results into table...")
            goods[l] = True
            dv_abs = popts[abstype][1]
            dv_abs_err = perrs[abstype][1]
            w_abs = popts[abstype][2]  # FWHM is 3rd parameter
            w_abs_err = perrs[abstype][2]
            # Calculate EW directly from the samples to get proper error estimation
            ew_samps = np.array(psamps[abstype])[:,0] / np.array(psamps[abstype])[:,3]  # EW = flux / continuum level
            ew_abs = np.median(ew_samps)
            ew_abs_err = (np.percentile(ew_samps, 84.0) - np.percentile(ew_samps, 16.0)) / 2.0
            megatable[f"DV_{abstype}_ABS"][i] = dv_abs
            megatable[f"DV_{abstype}_ABS_ERR"][i] = dv_abs_err
            megatable[f"W_{abstype}_ABS"][i] = w_abs
            megatable[f"W_{abstype}_ABS_ERR"][i] = w_abs_err
            megatable[f"EW_{abstype}_ABS"][i] = ew_abs
            megatable[f"EW_{abstype}_ABS_ERR"][i] = ew_abs_err
            if abstype == 'LI':
                megatable[f"DV_{abstype}_ABS_LINES"][i] = '+'.join(lispecs[best_lispec][-1])
            elif abstype == 'TOT':
                megatable[f"DV_{abstype}_ABS_LINES"][i] = '+'.join(lilines + hilines)
            print("Done.\n")
    count += any(goods)
    print(f"Found absorption in {str(count)}/{str(i+1)} sources.\n")


In [ ]:
megatable[((megatable['DV_LI_ABS'] - 3 * megatable['DV_LI_ABS_ERR'] > 0)
           | (megatable['DV_HI_ABS'] - 3 * megatable['DV_HI_ABS_ERR'] > 0)
           | (megatable['DV_TOT_ABS'] - 3 * megatable['DV_TOT_ABS_ERR'] > 0))]

In [ ]:
megatable[(megatable['DV_LI_ABS'] < 0) | (megatable['DV_HI_ABS'] < 0) | (megatable['DV_TOT_ABS'] < 0)]

In [ ]:
print(len(megatable[(megatable['DV_LI_ABS'] > -np.inf)]), 
      len(megatable[(megatable['DV_HI_ABS'] > -np.inf)]), 
      len(megatable[(megatable['DV_TOT_ABS'] > -np.inf)]))
print(len(megatable[~(megatable['DV_LI_ABS'] > -np.inf) * (megatable['DV_TOT_ABS'] > -np.inf)]), 
      len(megatable[(megatable['DV_HI_ABS'] < 0)]), 
      len(megatable[(megatable['DV_TOT_ABS'] < 0)]))

In [ ]:
#now save the table, which includes emission and absorption velocity offsets from lya:
megatable.write(f"./megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits", overwrite=True)